# ESG & Valuation — cross-tool validation (pure Python)

**AF1204 portfolio, self-exploration.** The main analysis (`pipelines/pipeline_4_merge.py`)
estimates `q ~ ESGscore + Firm_Size + Leverage` with industry and year fixed effects using
**statsmodels** (explicit dummy variables). Here the same Model 4 is re-estimated with
**linearmodels `PanelOLS`**, which *absorbs* the fixed effects with the within estimator —
a numerically different algorithm. If both agree to 4 decimals, we can be confident the
result is not an artifact of one implementation.

Runs in GitHub Codespaces with no network and no extra toolchain:
`pip install -r requirements.txt` then Run All.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../pipelines'))
from pipeline_5_crosscheck import load_provider_b, statsmodels_models, panelols_model4

df = load_provider_b()
print(f'Provider B panel: {len(df):,} firm-year rows')
df.head()

Provider B panel: 7,895 firm-year rows


,source,instrument,name,sector,q,ESGscore,Firm_Size,Leverage,Year,Industry
954,Provider_B,A,Agilent Technologies Inc,Manufacturing,2.066368,28.492539,8.747829,0.407224,2003,3825
955,Provider_B,A,Agilent Technologies Inc,Manufacturing,1.892605,52.431536,8.861634,0.322219,2004,3825
956,Provider_B,A,Agilent Technologies Inc,Manufacturing,2.384984,62.448436,8.817446,0.000000,2005,3825
957,Provider_B,A,Agilent Technologies Inc,Manufacturing,2.174623,69.633374,8.905037,0.411184,2006,3825
958,Provider_B,A,Agilent Technologies Inc,Manufacturing,2.081215,69.633374,8.929833,0.645331,2007,3825


In [2]:
sm_models = statsmodels_models(df)
sm4 = sm_models['Model 4']
sm4.params[['ESGscore', 'Firm_Size', 'Leverage']]

ESGscore     0.002650
Firm_Size   -0.513536
Leverage     0.009100
dtype: float64

In [3]:
pl4 = panelols_model4(df)
pl4.params[['ESGscore', 'Firm_Size', 'Leverage']]

ESGscore     0.002650
Firm_Size   -0.513536
Leverage     0.009100
Name: parameter, dtype: float64

In [4]:
import pandas as pd
terms = ['ESGscore', 'Firm_Size', 'Leverage']
check = pd.DataFrame({
    'statsmodels (dummies)': [sm4.params[t] for t in terms],
    'linearmodels (within)': [pl4.params[t] for t in terms],
}, index=terms)
check['match to 4dp'] = (check.iloc[:, 0] - check.iloc[:, 1]).abs() < 5e-5
check

,statsmodels (dummies),linearmodels (within),match to 4dp
ESGscore,0.002650,0.002650,True
Firm_Size,-0.513536,-0.513536,True
Leverage,0.009100,0.009100,True


## Publication-style table (Python `stargazer` package)
The same table family the course's Week 10 material uses, generated from the
statsmodels fits. This is what `data/regression_table.html` contains.

In [5]:
from stargazer.stargazer import Stargazer
from IPython.display import HTML
table = Stargazer([sm_models[k] for k in ('Model 1', 'Model 2', 'Model 3', 'Model 4')])
table.title("ESG score and firm valuation (Tobin's q) — Provider B")
table.covariate_order(terms)
table.add_line('Industry Indicators', ['No', 'No', 'Yes', 'Yes'])
table.add_line('Year Indicators', ['No', 'No', 'No', 'Yes'])
HTML(table.render_html())

**Conclusion.** Both implementations give ESGscore = **0.0026** (Model 4, N = 7,895):
the small positive within-industry-year ESG premium is not an implementation artifact.
Robustness caveats (provider choice, sub-samples) are discussed on the site's ESG tab.